TO BE RUN IN GOOGLE COLAB ON T4 GPU RUNTIME.

Installs VERSA toolkit and runs SRMR, NISQA and DNSMOS on the audio files.

Upon completion, the results file will be downloaded to the local downloads folder. This file must be copied to project/data/results in order to run the analysis notebook.

In [1]:
# clone versa into colab
!git clone https://github.com/wavlab-speech/versa.git

# change working directory to cloned versa
%cd versa

# install versa 
!pip install -e .

%cd ..

Cloning into 'versa'...
remote: Enumerating objects: 3441, done.
remote: Counting objects: 100% (796/796), done.
remote: Compressing objects: 100% (157/157), done.
remote: Total 3441 (delta 727), reused 639 (delta 639), pack-reused 2645 (from 3)
Receiving objects: 100% (3441/3441), 9.06 MiB | 14.93 MiB/s, done.
Resolving deltas: 100% (2355/2355), done.
/content/versa
Obtaining file:///content/versa
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Cloning https://github.com/ftshijt/espnet.git (to revision espnet_inference) to /tmp/pip-install-sjdfu97b/espnet_246968eeb9e747c8afa0e1bb361d7e59
  Running command git clone --filter=blob:none --quiet https://github.com/ftshijt/espnet.git /tmp/pip-install-sjdfu97b/espnet_246968eeb9e747c8afa0e1bb361d7e59
  Running command git checkout -b espnet_inference --track origin/espnet_inference


In [ ]:
import zipfile
from google.colab import files
import torchaudio

In [2]:
# UPLOAD split.zip from project/data/trimmed

%mkdir /content/versa/data/
%cd /content/versa/data/

uploaded = files.upload()

/content/versa/data


Saving split.zip to split.zip


In [3]:
zipname = "split.zip"

with zipfile.ZipFile(zipname, "r") as zip_ref:
    zip_ref.extractall()


In [4]:
%cd ../

/content/versa


In [5]:
# change versa files that cause further dependency issue
# this doesn't affect models themselves, just io

path = '/usr/local/lib/python3.12/dist-packages/s3prl/upstream/byol_s/byol_a/common.py'

with open(path, 'r') as f:
  content = f.read()

content = content.replace('torchaudio.set_audio_backend("sox_io")', '# torchaudio.set_audio_backend("sox_io")')

with open(path, 'w') as f:
  f.write(content)

path = '/usr/local/lib/python3.12/dist-packages/s3prl/upstream/mos_prediction/expert.py'

with open(path, 'r') as f:
  content = f.read()

content = content.replace('from torchaudio.sox_effects import apply_effects_tensor', '# from torchaudio.sox_effects import apply_effects_tensor')

with open(path, 'w') as f:
  f.write(content)



The next 3 cells download weights/install models that are used in this project.

NISQA and SRMR are not preinstalled by default with VERSA, but installation files are given.

In [6]:
!bash /content/versa/tools/setup_nisqa.sh

Cloning into 'NISQA'...
remote: Enumerating objects: 258, done.
remote: Counting objects: 100% (70/70), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 258 (delta 55), reused 45 (delta 45), pack-reused 188 (from 1)
Receiving objects: 100% (258/258), 2.27 MiB | 6.79 MiB/s, done.
Resolving deltas: 100% (130/130), done.


In [7]:
!mkdir -p /content/versa/tools/NISQA/weights

!wget https://github.com/gabrielmittag/NISQA/raw/master/weights/nisqa.tar -O /content/versa/tools/NISQA/weights/nisqa.tar

--2026-04-26 11:22:08--  https://github.com/gabrielmittag/NISQA/raw/master/weights/nisqa.tar
Resolving github.com (github.com)... 140.82.113.3
Connecting to github.com (github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/gabrielmittag/NISQA/master/weights/nisqa.tar [following]
--2026-04-26 11:22:09--  https://raw.githubusercontent.com/gabrielmittag/NISQA/master/weights/nisqa.tar
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1051663 (1.0M) [application/octet-stream]
Saving to: ‘/content/versa/tools/NISQA/weights/nisqa.tar’

/content/versa/tool 100%[===================>]   1.00M  --.-KB/s    in 0.03s   

2026-04-26 11:22:09 (29.7 MB/s) - ‘/content/versa/tools/NISQA/weights/

In [8]:
!bash /content/versa/tools/install_srmr.sh

Cloning into 'SRMRpy'...
remote: Enumerating objects: 157, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 157 (delta 38), reused 35 (delta 34), pack-reused 108 (from 1)
Receiving objects: 100% (157/157), 53.94 KiB | 876.00 KiB/s, done.
Resolving deltas: 100% (78/78), done.
Obtaining file:///content/versa/SRMRpy
  Preparing metadata (setup.py) ... done
     - 59.4 MB 50.5 MB/s 0:00:04
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 9.9 MB/s eta 0:00:00
  Created wheel for Gammatone: filename=Gammatone-1.0-py3-none-any.whl size=21760 sha256=85a5f04a8d8273d2e4b79ae7b56d6d410c13b8cc606666d2de964466fe6ceb47
  Stored in directory: /tmp/pip-ephem-wheel-cache-ko0pnh78/wheels/24/aa/ba/e5b0f55aecb3fe2d992c0b4a2cb457e116e62a6402b5847a5a
Successfully built Gammatone
  Running setup.py develop for SRMRpy


In [9]:
# creates config for VERSA and runs models

input_folder = "/content/versa/data/split/"
output_file = "/content/versa/versa_outputs_split.jsonl"

config_path = "/content/versa/config.yaml"

config_text = """
- name: pseudo_mos
  predictor_types:
    - dnsmos
    - utmos
  predictor_args:
    dnsmos:
      fs: 16000
- name: srmr
- name: nisqa
"""
# scoreq done in separate notebook because of dependency issue

with open(config_path, "w") as f:
    f.write(config_text)

!python /content/versa/versa/bin/scorer.py \
        --score_config "{config_path}" \
        --pred "{input_folder}" \
        --output_file "{output_file}" \
        --io dir \
        --use_gpu True

print(f"ANALYSIS FINISHED. OUTPUTS WRITTEN TO {output_file}")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
Failed to import Flash Attention, using ESPnet default: No module named 'flash_attn'
2026-04-26 11:22:44.065128: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777202564.086723    9170 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777202564.093665    9170 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777202564.111614    9170 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777202564.111661    9170 computation_placer.cc:177] comput

In [10]:
# downloads results file

files.download(output_file)

print('Results file downloaded!')
print('Please move this file from Downloads into project/data/results')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>